# Editability structure: on-manifold edits vs off-manifold artifact

**Question.** Probe-inversion edits change the decoded readout but barely change
the generated observations. Is that a *structural* fact (the decoded direction
is not a generative one) or a *mundane off-manifold artifact* (the min-norm
edit leaves the visited-state manifold and the dynamics project it away)?

**Disambiguation.** Build a linear approximation of the state manifold (PCA of
visited states), then compare three rollouts from the edit frame: unsteered,
pseudoinverse-steered (off-manifold-permitted), and manifold-constrained
(edit projected back onto the manifold via alternating projection). Instrument
with: off-manifold residual, readout *persistence* across the rollout, and
observation change. Then a generative-sensitivity sweep perturbs along the
probe direction vs PCA / random directions and measures observation change.

First pass: linear probe only (the editor is abstracted via `edit_fn`, so the
MLP/gradient editor can be dropped in later).

In [ ]:
import sys
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> import helpers

from dataclasses import replace

import numpy as np
import torch
import torch.autograd.functional as AF
import matplotlib.pyplot as plt
from IPython.display import display

import pim.eval as eval
import pim.figures as figs
from pim.extractors import LinearExtractor, StateDefinition, ProbeSpec, identity_mse, hungarian_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, project_to_subspace, offmanifold_residual, manifold_steer,
    fit_local_subspace, manifold_steer_local,
)
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader
from pim.figures.theme import style_ax_dark
from pim.simulator.viz import _BG_HEX as _DARK_BG, _TEXT_COLOR as _DARK_TXT
import helpers.nb_viz as nb_viz

In [ ]:
CHECKPOINT_PATH = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda"
BATCH_SIZE      = 512
NUM_WORKERS     = 6

N_OBJ           = 2          # objects to probe / steer
USE_HUNGARIAN   = False      # fixed reflectivities -> identity matching

N_CTRL          = 500        # edit samples to evaluate
CTRL_N_ROLLOUT  = 15         # rollout length post-edit
SUBSPACE_VAR    = 0.70       # variance kept by the GLOBAL state-manifold PCA subspace
POCS_ITERS      = 50         # edit<->project alternations for the manifold edit

# local tangent-PCA manifold (curvature-aware; global PCA residual is blind)
LOCAL_K         = 512        # nearest neighbors defining each local tangent patch
LOCAL_VAR       = 0.70       # variance kept WITHIN the local patch
LOCAL_BANK_SIZE = 50_000     # visited-state bank subsample used for the kNN

# visualization / sample selection
VIZ_MODE        = "top_obs_change"   # first | random | top_obs_change | top_edit
VIZ_N           = 3
VIZ_SEED        = 0

---
## 1 — Setup: model, probe, baselines

In [ ]:
model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

obs_baselines = eval.compute_obs_baselines(test.obs, test.clean_obs, test.obs_noise_std)
pos_baselines = eval.compute_pos_baselines(test.positions, test.position_noise_std)

# Teacher-force the whole test set -> hidden states (the bank of *visited* states).
preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)

# Fit the linear position probe.
state_def = StateDefinition(name="positions", state_shape=(N_OBJ, 2),
                            extract_fn=lambda b: b["positions"])
env_states_tf = test.positions[:, :-1, :N_OBJ, :]
vis_mask_tf   = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
loss_fn       = hungarian_mse if USE_HUNGARIAN else identity_mse

linear = LinearExtractor(model.hidden_size, state_def, use_lstsq=True)
train_mse = linear.fit(states_tf, env_states_tf, mask=vis_mask_tf, loss_fn=loss_fn, device=DEVICE)
linear = linear.to(DEVICE).eval()

print(f"Model   : {ckpt_info.run_name}  (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})")
print(f"Hidden  : {model.hidden_size}   states_tf={states_tf.shape}")
print(f"Probe   : linear position, train MSE = {train_mse:.6f}")

---
## 2 — State manifold (PCA of visited states)

The visited hidden states are assumed to lie near a low-dim affine subspace. A
min-norm probe edit ignores this subspace; the **off-manifold residual**
(`‖h − project(h)‖`) measures how far an edit strays relative to real states.

In [ ]:
# Fit the manifold subspace on all visited (teacher-forced) states.
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(
    subspace,
    mean=subspace.mean.to(DEVICE),
    basis=subspace.basis.to(DEVICE),
    explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE),
)
print(f"kept {subspace.n_components}/{subspace.hidden_size} components "
      f"({subspace.total_explained:.4f} variance)")

# Full spectrum for the scree plot.
full = fit_state_subspace(states_tf, n_components=model.hidden_size)
cum = torch.cumsum(full.explained_variance_ratio, 0).cpu().numpy()

# Residual scale of *real* states (the on-manifold reference).
real_res = offmanifold_residual(
    torch.from_numpy(states_tf.reshape(-1, model.hidden_size)[:5000]).float(), subspace
)
print(f"off-manifold residual of real states: mean={real_res.mean():.4f}  "
      f"p95={np.percentile(real_res.numpy(), 95):.4f}")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(np.arange(1, len(cum) + 1), cum, marker=".", ms=4)
ax.axhline(SUBSPACE_VAR, color="0.6", ls="--", lw=1)
ax.axvline(subspace.n_components, color="C3", ls="--", lw=1, label=f"k={subspace.n_components}")
ax.set_xlabel("# components"); ax.set_ylabel("cumulative variance")
ax.set_title("State-manifold PCA spectrum"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); display(fig); plt.close(fig)

---
## 3 — Experiment 1: on-manifold vs off-manifold edits

Four edits at `edit_frame`, then a free rollout:
- **unsteered** — control
- **pseudoinverse** — min-norm edit, off-manifold permitted (the original method)
- **manifold** (global) — projected onto the global-PCA subspace (alternating projection)
- **local** — projected onto a *local tangent-PCA* patch (curvature-aware; the
  honest on-manifold edit, since global-PCA residual is blind to in-subspace moves)

Then a **reversion-vs-drift** diagnostic tracks where the decoded position goes:
toward the original/pre-edit (reversion), toward the moving post-edit GT (success),
or away from everything (drift). If even the local edit reverts and never moves the
output, the result is **structural**; if it starts tracking GT, the original
failure was the **off-manifold artifact**.

In [ ]:
N = min(N_CTRL, edits.n_samples)
warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame,
                            n_viz=N, n_ctx_show=8, device=DEVICE)  # n_viz=N: pre-edit ctx for all

# Steering target = GT positions at the edit frame (flattened).
targets = edits.positions[:N, edits.edit_frame, :N_OBJ, :].reshape(N, N_OBJ * 2)

A, b, A_pinv = probe_decomposition(linear)            # on DEVICE (probe is on DEVICE)
h0  = torch.from_numpy(warm.h_at_edit).float().to(DEVICE)
tgt = torch.from_numpy(targets).float().to(DEVICE)
edit_fn = lambda h, t: inject_state(h, t, A, A_pinv, b)

# On-device bank for local tangent-PCA (subsample the visited states once so the
# per-sample kNN doesn't re-transfer the bank to GPU on every call).
_bank_all = states_tf.reshape(-1, model.hidden_size)
_sub = np.random.RandomState(0).choice(
    _bank_all.shape[0], size=min(LOCAL_BANK_SIZE, _bank_all.shape[0]), replace=False)
bank_dev = torch.from_numpy(_bank_all[_sub]).float().to(DEVICE)

h_pinv     = inject_state(h0, tgt, A, A_pinv, b)                                  # off-manifold OK
h_manifold = manifold_steer(h0, tgt, edit_fn, subspace_dev, n_iters=POCS_ITERS)   # global manifold
h_local    = manifold_steer_local(h0, tgt, edit_fn, bank_dev,                     # local tangent
                                   k_neighbors=LOCAL_K, n_iters=POCS_ITERS,
                                   var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)

def _readout_rmse(h):
    return float(((h @ A.T + b) - tgt).pow(2).mean().sqrt())
def _resid_global(h):
    return float(offmanifold_residual(h, subspace_dev).mean())
def _resid_local(h, n_probe=100):  # per-sample local subspace residual (the honest one)
    res = []
    for i in range(min(n_probe, h.shape[0])):
        sub = fit_local_subspace(bank_dev, h[i], k_neighbors=LOCAL_K,
                                 var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
        res.append(float(offmanifold_residual(h[i:i + 1], sub).mean()))
    return float(np.mean(res))

print(f"{'edit':12s} {'readout RMSE':>13s} {'global resid':>13s} {'local resid':>12s}")
for name, h in [("unsteered", h0), ("pseudoinv", h_pinv),
                ("manifold", h_manifold), ("local", h_local)]:
    print(f"{name:12s} {_readout_rmse(h):13.4f} {_resid_global(h):13.4f} {_resid_local(h):12.4f}")
print("\nGlobal resid is blind to edits that move within the kept subspace; the")
print("LOCAL resid is the honest off-manifold detector (curvature-aware). High")
print("manifold/local readout RMSE => target position is unreachable on the manifold.")

In [ ]:
@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    """Roll out the model from each flat state; step 0 = decode (no advance)."""
    obs_all, h_all = [], []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, hs = _rollout(model, h, n_rollout)
        obs_all.append(o); h_all.append(hs)
    return np.stack(obs_all), np.stack(h_all)

obs_u, hs_u = rollout_from_flat(warm.h_at_edit,                  CTRL_N_ROLLOUT)  # unsteered
obs_p, hs_p = rollout_from_flat(h_pinv.detach().cpu().numpy(),   CTRL_N_ROLLOUT)  # pseudoinverse
obs_m, hs_m = rollout_from_flat(h_manifold.detach().cpu().numpy(), CTRL_N_ROLLOUT)  # global manifold
obs_l, hs_l = rollout_from_flat(h_local.detach().cpu().numpy(),  CTRL_N_ROLLOUT)  # local manifold
print("rollouts:", obs_u.shape, obs_p.shape, obs_m.shape, obs_l.shape)

In [ ]:
gt_obs = edits.obs[:N, edits.edit_frame:edits.edit_frame + CTRL_N_ROLLOUT]
gt_positions = edits.positions[:N, edits.edit_frame:edits.edit_frame + CTRL_N_ROLLOUT, :N_OBJ, :]

# --- observation-space signals (no probe needed) ---
def rms_step(a, ref):  # rms over samples & rays, per rollout step
    return np.sqrt(((a - ref) ** 2).mean(axis=(0, 2)))

variants_obs = {"pseudoinv": obs_p, "manifold": obs_m, "local": obs_l}
chg = {k: rms_step(v, obs_u)  for k, v in variants_obs.items()}   # effect on the output
err = {"unsteered": rms_step(obs_u, gt_obs)}
err.update({k: rms_step(v, gt_obs) for k, v in variants_obs.items()})  # error vs post-edit GT

# --- probe persistence: decode position from the *generated* hidden states ---
@torch.no_grad()
def decode_pos(h_array):
    t = torch.as_tensor(h_array, dtype=torch.float32, device=DEVICE)
    return linear(t).cpu().numpy()    # (N, n_rollout, N_OBJ, 2)

pos_u, pos_p, pos_m, pos_l = (decode_pos(hs_u), decode_pos(hs_p),
                              decode_pos(hs_m), decode_pos(hs_l))
tgt_np = targets.reshape(N, N_OBJ, 2)

def dist_to_target(pos):  # mean over samples & objects, per step
    return np.sqrt(((pos - tgt_np[:, None]) ** 2).sum(-1)).mean(axis=(0, 2))

d_target = {k: dist_to_target(p) for k, p in
            {"pseudoinv": pos_p, "manifold": pos_m, "local": pos_l}.items()}
steps = np.arange(CTRL_N_ROLLOUT)
COL = {"unsteered": "C0", "pseudoinv": "C1", "manifold": "C2", "local": "C3"}

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

ax = axes[0, 0]
labels = ["real", "unsteered", "pseudoinv", "manifold", "local"]
ax.bar(labels,
       [float(real_res.mean()), _resid_global(h0), _resid_global(h_pinv),
        _resid_global(h_manifold), _resid_global(h_local)],
       color=["0.6", "C0", "C1", "C2", "C3"])
ax.set_ylabel("global off-manifold residual")
ax.set_title("(a) off-manifold (global PCA — blind to in-subspace edits)")
ax.tick_params(axis="x", labelrotation=20)

ax = axes[0, 1]
ax.plot(steps, dist_to_target(pos_u), "C0--", label="unsteered")
for k, d in d_target.items():
    ax.plot(steps, d, marker="o", ms=3, color=COL[k], label=k)
ax.set_xlabel("rollout step"); ax.set_ylabel("decoded pos. dist to target")
ax.set_title("(b) readout persistence (lower = edit holds)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
for k, c in chg.items():
    ax.plot(steps, c, marker="o", ms=3, color=COL[k], label=k)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS obs change vs unsteered")
ax.set_title("(c) did the edit move the output?"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
for k, e in err.items():
    ax.plot(steps, e, marker="o", ms=3, color=COL[k], label=k)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS obs error vs GT")
ax.set_title("(d) error vs post-edit ground truth"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.tight_layout(); display(fig); plt.close(fig)

print("mean obs change vs unsteered:  " + "  ".join(f"{k}={v.mean():.4f}" for k, v in chg.items()))
print("mean obs error vs GT:          " + "  ".join(f"{k}={v.mean():.4f}" for k, v in err.items()))

In [ ]:
# Sample selection for the waterfalls / per-sample viz (swap VIZ_MODE in config).
def select_samples(mode=VIZ_MODE, n=VIZ_N, seed=VIZ_SEED):
    if mode == "first":
        return np.arange(min(n, N))
    if mode == "random":
        return np.random.RandomState(seed).choice(N, size=min(n, N), replace=False)
    if mode == "top_obs_change":              # samples the global manifold edit moved most
        score = ((obs_m - obs_u) ** 2).mean(axis=(1, 2))
        return np.argsort(score)[::-1][:n]
    if mode == "top_edit":                    # largest GT teleport at the edit frame
        ef = edits.edit_frame
        disp = np.linalg.norm(
            edits.positions[:N, ef, :N_OBJ] - edits.positions[:N, ef - 1, :N_OBJ], axis=-1
        ).max(axis=1)
        return np.argsort(disp)[::-1][:n]
    raise ValueError(f"unknown VIZ_MODE: {mode}")

viz_idx = select_samples()
print(f"VIZ_MODE={VIZ_MODE!r}  ->  samples {list(map(int, viz_idx))}")

In [ ]:
# Reversion vs drift: where does the decoded position go after the edit?
# Anchors: injected TARGET (static) | post-edit GT trajectory (MOVING — so normal
# disc motion is not mislabeled as drift) | UNSTEERED decoded (original) | PRE-EDIT pos.
pre_pos = edits.positions[:N, edits.edit_frame - 1, :N_OBJ, :]   # (N, N_OBJ, 2)

def d_to(pos, anchor):   # anchor broadcastable to (N, n_rollout, N_OBJ, 2)
    return np.sqrt(((pos - anchor) ** 2).sum(-1)).mean(axis=(0, 2))

anchors = {
    "target (static)":       tgt_np[:, None],
    "post-edit GT (moving)": gt_positions,
    "unsteered decoded":     pos_u,
    "pre-edit pos":          pre_pos[:, None],
}
variants_pos = {"pseudoinv": pos_p, "manifold": pos_m, "local": pos_l}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (vname, pos) in zip(axes, variants_pos.items()):
    for aname, anc in anchors.items():
        ax.plot(steps, d_to(pos, anc), marker=".", ms=4, label=aname)
    ax.set_title(vname); ax.set_xlabel("rollout step"); ax.grid(alpha=0.3)
axes[0].set_ylabel("decoded-position distance")
axes[0].legend(fontsize=8, title="distance to")
fig.suptitle("Reversion vs drift — toward original/pre-edit = reversion; "
             "toward moving GT = success; away from all = drift")
fig.tight_layout(); display(fig); plt.close(fig)

---
## 4 — Experiment 2: generative-sensitivity sweep

Perturb the warmed-up state along a direction, roll out, measure observation
change (no probe needed for the outcome). Two views: **(1)** magnitudes in units of
data-std along each direction (realistic moves), and **(2)** at **matched absolute
‖Δh‖** — this removes the confound that probe directions have small data-std and
were thus under-stepped in view (1).

Structural signature: the **probe** directions are as flat as **random**, while
**PCA** directions the dynamics traverse produce large change.

The final cell is the **analytic** counterpart: the decoder Jacobian
`J = ∂decode/∂h`. We check how much of each probe direction lies in the decoder's
high-sensitivity (top-singular) subspace — low (≈ random) confirms decode≠generate
without any rollout.

In [ ]:
n_sweep      = 64
n_roll_sweep = 10
h_base = warm.h_at_edit[:n_sweep]

Xc = torch.from_numpy(states_tf.reshape(-1, model.hidden_size)).float()
Xc = Xc - Xc.mean(0)

def unit(v):
    return v / v.norm()

# Directions (all unit norm). Probe rows = the decode directions; PCA = directions
# the dynamics actually traverse; random = null floor.
dirs = {
    "probe obj0-x": unit(A[0].detach().cpu()),
    "probe obj0-y": unit(A[1].detach().cpu()),
    "PCA #1":       subspace.basis[:, 0].detach().cpu(),
    "PCA #2":       subspace.basis[:, 1].detach().cpu(),
    "random":       unit(torch.randn(model.hidden_size, generator=torch.Generator().manual_seed(0))),
}
COLW = {"probe obj0-x": "C1", "probe obj0-y": "C4", "PCA #1": "C2",
        "PCA #2": "C0", "random": "0.6"}

def sigma(d):  # data std along a unit direction
    return float((Xc @ d).std())

base_obs, _ = rollout_from_flat(h_base, n_roll_sweep)

# (1) sigma-scaled: magnitude in units of data std along each direction ("realistic move").
mults = [0.0, 1.0, 2.0, 4.0]
sweep_sigma = {}
for name, d in dirs.items():
    s = sigma(d); dd = d.numpy()
    sweep_sigma[name] = [np.sqrt(((rollout_from_flat(h_base + (k * s) * dd, n_roll_sweep)[0]
                                   - base_obs) ** 2).mean()) for k in mults]

# (2) matched ABSOLUTE ||Δh||: removes the per-direction-sigma confound (probe dirs
#     have small sigma, so sigma-scaling under-steps them). dirs are unit, so ||a*d|| = a.
abs_norms = [0.0, 0.5, 1.0, 2.0, 4.0]
sweep_abs = {}
for name, d in dirs.items():
    dd = d.numpy()
    sweep_abs[name] = [np.sqrt(((rollout_from_flat(h_base + a * dd, n_roll_sweep)[0]
                                 - base_obs) ** 2).mean()) for a in abs_norms]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for name, row in sweep_sigma.items():
    axes[0].plot(mults, row, marker="o", color=COLW[name], label=name)
axes[0].set_xlabel("magnitude (data std along direction)")
axes[0].set_title("(1) σ-scaled (realistic moves)")
for name, row in sweep_abs.items():
    axes[1].plot(abs_norms, row, marker="o", color=COLW[name], label=name)
axes[1].set_xlabel("absolute ‖Δh‖")
axes[1].set_title("(2) matched absolute magnitude")
for ax in axes:
    ax.set_ylabel("RMS observation change"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("Generative sensitivity by direction "
             "(probe ≈ random ≪ PCA ⇒ decode direction is not generative)")
fig.tight_layout(); display(fig); plt.close(fig)

print("σ along each direction: " + "  ".join(f"{n}={sigma(d):.3f}" for n, d in dirs.items()))

In [ ]:
# Decoder-Jacobian alignment: the ANALYTIC counterpart of the sensitivity sweep.
# J = d(decode)/dh  (R x H). Its top right-singular vectors span the h-directions the
# decoder is most sensitive to (its "generative" directions). We measure how much of
# each probe readout direction lies in that subspace. Low (≈ random) => the decoder
# ignores the decode direction => decode != generate, confirmed without any rollout.
def decode_from_flat(hvec):
    return model.decode(model.state_from_flat(hvec.unsqueeze(0))).squeeze(0)  # (R,)

R_KEEP = 8            # decoder-sensitive subspace rank to test against
n_jac  = 32
samp   = np.random.RandomState(0).choice(N, size=min(n_jac, N), replace=False)

probe_dirs = torch.stack([unit(A[i].detach().cpu()) for i in range(A.shape[0])])   # (D, H)
_g = torch.Generator().manual_seed(1)
rand_dirs = torch.stack([unit(torch.randn(model.hidden_size, generator=_g))
                         for _ in range(A.shape[0])])

def proj_frac(dirs, Vr):   # mean squared projection of unit dirs onto span(Vr)
    P = dirs @ Vr          # (k, r)
    return float((P ** 2).sum(dim=1).mean())

pf_probe, pf_rand, energy = [], [], []
for i in samp:
    h = torch.from_numpy(warm.h_at_edit[i]).float().to(DEVICE).requires_grad_(True)
    J = AF.jacobian(decode_from_flat, h).detach().cpu()          # (R, H)
    _, S, Vh = torch.linalg.svd(J, full_matrices=False)
    Vr = Vh[:R_KEEP].T                                           # (H, R_KEEP)
    pf_probe.append(proj_frac(probe_dirs, Vr))
    pf_rand.append(proj_frac(rand_dirs, Vr))
    energy.append(float((S[:R_KEEP] ** 2).sum() / (S ** 2).sum()))

print(f"decoder Jacobian: top-{R_KEEP} singular dirs capture "
      f"{np.mean(energy):.1%} of output sensitivity")
print("mean squared projection onto that decoder-sensitive subspace "
      "(1.0 = fully inside, low = ignored):")
print(f"  probe  directions: {np.mean(pf_probe):.3f} ± {np.std(pf_probe):.3f}")
print(f"  random directions: {np.mean(pf_rand):.3f} ± {np.std(pf_rand):.3f}")
print("probe ≈ random ⇒ the decoder does not read position from the probe direction.")

---
## 5 — Positive control + waterfalls

**Swap control:** roll out from a *different real* warmed-up state. This must change
the observations a lot — it proves the rollout responds to (valid) state changes, so
a small edit-induced change is meaningful, not a dead pipeline.

**Waterfalls** (4-panel: GT | unsteered | manifold | local) for the samples chosen by
`VIZ_MODE` (`top_obs_change` surfaces the most-affected edits, not faint defaults).

**PCA-component explorer:** roll out from `h ± α·σ·PCᵢ` to *see* what each principal
direction encodes generatively — swap `PC_INDEX` / `PC_ALPHA` / `PC_SAMPLE`.

In [ ]:
# Reusable N-panel grayscale waterfall (generalizes figs.plot_controllability_waterfalls).
def plot_waterfall_grid(panels, titles, *, edit_frame=None, suptitle=""):
    n = len(panels)
    fig = plt.figure(figsize=(4.4 * n, 5.0), facecolor=_DARK_BG)
    if suptitle:
        fig.suptitle(suptitle, color=_DARK_TXT, fontsize=11, y=0.99)
    for k, (img, ttl) in enumerate(zip(panels, titles)):
        ax = fig.add_subplot(1, n, k + 1); style_ax_dark(ax)
        ax.imshow(np.clip(img, 0, 1), aspect="auto", origin="upper",
                  interpolation="nearest", cmap="gray", vmin=0, vmax=1)
        if edit_frame is not None:
            ax.axhline(edit_frame - 0.5, color="#fa8850", lw=1.2, ls="--", alpha=0.7)
        ax.set_title(ttl, color=_DARK_TXT, fontsize=10)
        ax.set_xlabel("ray", color=_DARK_TXT, fontsize=9)
        ax.set_ylabel("frame", color=_DARK_TXT, fontsize=9)
    fig.tight_layout(); return fig

# --- positive control: swap in another sample's real state ---
perm = np.random.RandomState(0).permutation(N)
obs_swap, _ = rollout_from_flat(warm.h_at_edit[perm], CTRL_N_ROLLOUT)
chg_swap = rms_step(obs_swap, obs_u)
print("mean RMS obs change vs unsteered:")
print(f"  swap (real other state) = {chg_swap.mean():.4f}   <- pipeline responds this much")
for k, v in chg.items():
    print(f"  {k:22s}= {v.mean():.4f}")

# Decoded pre-edit context for the trajectory plots (needs n_viz=N in the warm-up cell).
pre_dec = decode_pos(warm.h_pre_edit)        # (N, n_ctx_show, N_OBJ, 2)
# Single-probe spec so we can reuse figs.plot_controllability_trajectory (manifold edit only).
traj_probe = [ProbeSpec(name="decoded", probe=linear, marker="o", color_idx=0, linestyle="-")]

# --- per selected sample: waterfalls + decoded-position trajectory (manifold vs unsteered) ---
for i in viz_idx:
    i = int(i)
    pre = edits.obs[i, :edits.edit_frame]
    full = lambda post: np.concatenate([pre, post], axis=0)
    fig = plot_waterfall_grid(
        [full(gt_obs[i]), full(obs_u[i]), full(obs_m[i]), full(obs_l[i])],
        ["GT", "unsteered", "manifold (global)", "local tangent"],
        edit_frame=edits.edit_frame,
        suptitle=f"Sample {i}  —  edit at frame {edits.edit_frame}  (VIZ_MODE={VIZ_MODE})",
    )
    display(fig); plt.close(fig)

    # Side-by-side x / y(depth) decoded positions: manifold-steered (solid) vs
    # unsteered (open), with GT line + faint pre-edit context. Manifold global only.
    fig = figs.plot_controllability_trajectory(
        pre_edit_gt=edits.positions[i, edits.edit_frame - warm.n_ctx_show:edits.edit_frame, :N_OBJ],
        pre_edit_decoded={"decoded": pre_dec[i]},
        post_edit_gt=gt_positions[i],
        steered_decoded={"decoded": pos_m[i]},
        probes=traj_probe,
        scene_colors=edits.colors[i, :N_OBJ],
        sample_idx=i, edit_frame=edits.edit_frame, n_rollout=CTRL_N_ROLLOUT,
        show_unsteered=True,
        unsteered_decoded={"decoded": pos_u[i]},
    )
    display(fig); plt.close(fig)

In [ ]:
# PCA-component edit explorer: what does each principal direction generate?
# Roll out from h ± alpha*sigma*PC_i and read the waterfall. Swap PC_INDEX / PC_ALPHA.
PC_INDEX  = 0                 # which principal component to edit along
PC_ALPHA  = 3.0               # magnitude in units of sigma along that PC
PC_SAMPLE = int(156) #viz_idx[0])   # which sample

pc   = subspace.basis[:, PC_INDEX]
s_pc = sigma(pc.cpu())        # data std along this PC (sigma() defined in the sweep cell)
pc_d = pc.cpu().numpy()
h_s  = warm.h_at_edit[PC_SAMPLE]
pre  = edits.obs[PC_SAMPLE, :edits.edit_frame]

def _roll1(hvec):
    o, _ = rollout_from_flat(hvec[None], CTRL_N_ROLLOUT)
    return np.concatenate([pre, o[0]], axis=0)

fig = plot_waterfall_grid(
    [_roll1(h_s - PC_ALPHA * s_pc * pc_d), _roll1(h_s), _roll1(h_s + PC_ALPHA * s_pc * pc_d)],
    [f"-{PC_ALPHA:g}σ·PC{PC_INDEX}", "unsteered", f"+{PC_ALPHA:g}σ·PC{PC_INDEX}"],
    edit_frame=edits.edit_frame,
    suptitle=f"Sample {PC_SAMPLE} — generative effect of PCA component {PC_INDEX}",
)
display(fig); plt.close(fig)